# Classify Job Postings

Run `3-analyse-aie-job-postings.ipynb` first so that `jobs/1-scraped_jobs.jsonl` exists.

This notebook reads the scraped jobs, uses an LLM to classify whether each role is truly an AI engineering role, and writes the accepted jobs to `jobs/2-classified-jobs.jsonl`.


In [1]:
import json
from pathlib import Path

import pandas as pd
from openai import OpenAI
from IPython.display import HTML, display

### Load scraped jobs from file

In [2]:
scraped_jobs_path = Path("jobs") / "1-scraped_jobs.jsonl"

if not scraped_jobs_path.exists():
    raise FileNotFoundError(
        f"Scraped jobs JSONL file not found: {scraped_jobs_path.resolve()}. Run 3-analyse-aie-job-postings.ipynb first."
    )

if scraped_jobs_path.stat().st_size == 0:
    raise ValueError(
        "The scraped jobs JSONL file is empty. Run 3-analyse-aie-job-postings.ipynb first."
    )

jobs_df = pd.read_json(scraped_jobs_path, lines=True)

print(f"Scraped jobs JSONL file loaded successfully with {len(jobs_df)} entries!")

Scraped jobs JSONL file loaded successfully with 139 entries!


### Define the prompt

We define the instructions that tell the model what counts as an AI engineering role.

In [3]:
client = OpenAI()

instructions = """
You classify whether a job posting is truly for an AI Engineering role.

AI Engineering definition:
- AI engineering means building applications on top of foundation models or in other words integrating them into products.
- Traditional ML engineering focuses on building, training, or tuning models; AI engineering primarily leverages existing models.
- MLOps and platform engineering are not AI engineering, as they focus on infrastructure and tooling rather than building AI-powered features.

Decision rules:
- Set is_ai_engineering_role to true when the main responsibility is building product or application features on top of foundation models or LLMs.
- Set is_ai_engineering_role to false when the role is mainly traditional software engineering, data science, analytics, ML research, model training, classical ML engineering, MLOps or platform work, or something else where AI application work is not the core responsibility.
- If the posting is ambiguous or unclear, set is_ai_engineering_role to false.
""".strip()

### Define the schema

We tell the model to return structured output with a boolean decision and a short reason.

In [4]:
schema = {
    "type": "object",
    "properties": {
        "is_ai_engineering_role": {"type": "boolean"},
        "reason": {"type": "string"},
    },
    "required": ["is_ai_engineering_role", "reason"],
    "additionalProperties": False,
}

### Make the LLM calls

We now ask the model to classify each job posting.

In [5]:
results = []

for i, (_, job) in enumerate(jobs_df.iterrows(), start=1):
    title = job["title"]
    description = job["description"]

    print(f"Screening job {i}/{len(jobs_df)}: {title}")

    response = client.responses.create(
        model="gpt-5.4-mini",
        instructions=instructions,
        input=f"""Classify this job posting.\n\nTitle: {title}\n\nDescription:\n{description}""",
        text={
            "format": {
                "type": "json_schema",
                "name": "ai_engineering_job_screening",
                "schema": schema,
                "strict": True,
            },
            "verbosity": "low",
        },
    )

    result = json.loads(response.output_text)
    is_ai_engineering_role = result["is_ai_engineering_role"]
    reason = result["reason"].strip()

    results.append(
        {
            "is_ai_engineering_role": is_ai_engineering_role,
            "llm_reason": reason,
        }
    )

Screening job 1/139: AI Software Engineer
Screening job 2/139: AI Software Engineer-Senior
Screening job 3/139: Senior AI/ML Engineer - Engineering Excellence (Full Stack Developer) Vice President
Screening job 4/139: Sr. Gen AI Engineer (USC/GC) W2
Screening job 5/139: Principal AI Engineer
Screening job 6/139: Principal Engineer, Agentic AI
Screening job 7/139: Java AI Engineer | Charlotte, NC, USA (3 days onsite) | Contract W2 only
Screening job 8/139: AI Engineer – Optical Design
Screening job 9/139: Senior AI Engineer
Screening job 10/139: Senior Adversarial AI Engineer
Screening job 11/139: Data and AI Engineer
Screening job 12/139: Sr. Data and AI Engineer
Screening job 13/139: Enterprise Automation & AI Engineer
Screening job 14/139: AI Engineer (Hybrid)
Screening job 15/139: VP, Agentic AI Engineering
Screening job 16/139: Senior AI Engineer (USA - Remote)
Screening job 17/139: Junior AI Engineer
Screening job 18/139: Founding AI Engineer
Screening job 19/139: AI Engineer, Voi

### Combine the results

We join the model output back to the scraped jobs and save the accepted jobs

In [6]:
classified_jobs_path = Path("jobs") / "2-classified-jobs.jsonl"

results_df = pd.DataFrame(results)
screened_jobs = pd.concat([jobs_df, results_df], axis=1)

ai_engineer_jobs = screened_jobs[screened_jobs["is_ai_engineering_role"]].copy()
non_ai_engineer_count = int((~screened_jobs["is_ai_engineering_role"]).sum())
ai_engineer_jobs.to_json(
    classified_jobs_path, orient="records", lines=True, force_ascii=False
)

### Display the results

In [7]:
print(f"Saved classified jobs to: {classified_jobs_path.resolve()}")
print(f"Jobs screened by LLM: {len(screened_jobs)}")
print(f"Jobs classified as AI engineering roles: {len(ai_engineer_jobs)}")
print(f"Jobs classified as not AI engineering or unclear: {non_ai_engineer_count}")

results_to_show = screened_jobs[
    ["title", "is_ai_engineering_role", "llm_reason", "job_url"]
].copy()
results_to_show["job_url"] = results_to_show["job_url"].apply(
    lambda url: f'<a href="{url}" target="_blank">{url}</a>' if pd.notna(url) else ""
)

display(HTML(results_to_show.to_html(escape=False, index=False)))

Saved classified jobs to: /Users/lukaslechner/PythonProjects/ai-engineering-foundations-labs/1-introduction/jobs/2-classified-jobs.jsonl
Jobs screened by LLM: 139
Jobs classified as AI engineering roles: 101
Jobs classified as not AI engineering or unclear: 38


title,is_ai_engineering_role,llm_reason,job_url
AI Software Engineer,True,"The role centers on building AI-powered application features using LLMs, including LLM orchestration, RAG, data preparation for LLM consumption, and fine-tuning/supporting AI tools. That matches AI engineering as product/application work on top of foundation models rather than traditional ML training or MLOps.",https://www.indeed.com/viewjob?jk=be2d926a6fa9a23a
AI Software Engineer-Senior,True,"The role centers on building AI-powered application tools using existing LLMs, including LLM integration, orchestration frameworks, RAG, data preparation for LLMs, and fine-tuning to support mission features. It is not primarily model research or training.",https://www.indeed.com/viewjob?jk=5541ba282fe752b1
Senior AI/ML Engineer - Engineering Excellence (Full Stack Developer) Vice President,True,"The role’s main focus is building and integrating GenAI/LLM and agentic AI features into enterprise applications, including prompt engineering, RAG, assistant tools, and production deployment. This is AI application development rather than model research, classical ML, or MLOps-only work.",https://www.indeed.com/viewjob?jk=9053791eafe28b07
Sr. Gen AI Engineer (USC/GC) W2,True,"The role’s main responsibility is building production-grade applications and features on top of foundation models/LLMs using OpenAI, LangChain, Hugging Face, and agentic workflows. That matches AI engineering rather than traditional ML, MLOps, or platform work.",https://www.indeed.com/viewjob?jk=7952ffbf91ec53d6
Principal AI Engineer,False,"This is primarily an AI/platform engineering role focused on infrastructure, developer tools, CI/CD, Kubernetes, cloud services, and MLOps-style lifecycle support. It does mention LLM integration and agentic AI, but building product features on top of foundation models is not the core responsibility.",https://www.indeed.com/viewjob?jk=f38b13ef5bd2358a
"Principal Engineer, Agentic AI",True,"The role’s core responsibility is building production agentic AI applications on top of foundation models (Copilot Studio, Azure AI Foundry, Azure OpenAI, RAG, multi-agent orchestration, integrations, evaluation, and deployment). It is not mainly model training, MLOps, or traditional ML engineering.",https://www.indeed.com/viewjob?jk=d53b0d21f106bf44
"Java AI Engineer | Charlotte, NC, USA (3 days onsite) | Contract W2 only",False,"The role is primarily Java/backend engineering with microservices, Kafka, APIs, databases, and cloud infrastructure. The AI content is limited to exposure to conversational AI/LLM concepts and prompt engineering, not building product features on top of foundation models as the core responsibility. Ambiguous and not clearly an AI engineering role.",https://www.indeed.com/viewjob?jk=f2e036639932b33a
AI Engineer – Optical Design,True,"The role is centered on building and iterating on agentic/LLM-integrated product systems: running experiments, debugging tool loops and multi-turn state, choosing architectures, and eventually owning production deployment on AWS. That fits AI engineering because it applies foundation models in an application context rather than training models or doing MLOps/platform work.",https://www.indeed.com/viewjob?jk=836a713564071627
Senior AI Engineer,True,"The role is centered on building and maintaining an enterprise conversational AI platform using Azure OpenAI, prompt engineering, RAG, semantic search, and integrations with business data sources. That matches AI engineering as product/application development on top of foundation models, not model training or MLOps.",https://www.indeed.com/viewjob?jk=16933d29b84666ae
Senior Adversarial AI Engineer,False,"The role is primarily AI security/adversarial analysis, red teaming, evaluation, and integration for government systems rather than building product features on top of foundation models. It is more aligned with AI assurance/security and applied research than core AI engineering 